In [1]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, roc_curve

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.config import ANOMALY_SIGMA, DATA_PROCESSED, HEALTHY_RATIO, IMS_DIR, IMS_FS, MODELS_DIR, OVERLAP, REPORTS_DIR, SEED, TRANSITION_RATIO, WINDOW_SIZE
from src.data.loader import load_ims_run
from src.features.extraction import extract_bearing_features
from src.models import compute_metrics
from src.visualization.plots import plot_comparison_table, plot_roc_curves

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"
METRICS_DIR = MODELS_DIR / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
baseline_metrics = pd.read_csv(METRICS_DIR / "baseline_metrics.csv")
lstm_metrics = pd.read_csv(METRICS_DIR / "lstm_ae_metrics.csv")
development_metrics = pd.concat([baseline_metrics, lstm_metrics], ignore_index=True)
best_row = development_metrics.sort_values("f1", ascending=False).iloc[0]
best_model = best_row["model"]
frozen_threshold = float(best_row["threshold"])
print(development_metrics[["model", "roc_auc", "pr_auc", "f1", "precision", "recall"]].to_string(index=False))

           model  roc_auc   pr_auc       f1  precision   recall
     Rolling RMS 0.999998 0.999995 0.999554   1.000000 0.999109
Isolation Forest 0.993394 0.963745 0.969618   0.945008 0.995544
   One-Class SVM 0.999796 0.999393 0.986784   0.975610 0.998217
LSTM Autoencoder 0.999371 0.998085 0.970809   0.943275 1.000000


In [5]:
femto_features = pd.read_parquet(DATA_PROCESSED / "femto_features_all_bearings.parquet")
bearing1_2 = femto_features[femto_features["bearing_id"] == "Bearing1_2"].copy()
bearing1_2 = bearing1_2.sort_values(["file_idx", "window_idx"]).reset_index(drop=True)

In [6]:
file_count = int(bearing1_2["file_idx"].max() + 1)
healthy_end = int(file_count * HEALTHY_RATIO)
anomaly_start = int(file_count * (HEALTHY_RATIO + TRANSITION_RATIO))
label_mask = (bearing1_2["file_idx"] < healthy_end) | (bearing1_2["file_idx"] >= anomaly_start)
bearing1_2_labeled = bearing1_2.loc[label_mask].copy()
bearing1_2_y = (bearing1_2_labeled["file_idx"] >= anomaly_start).astype(int).to_numpy()
bearing1_2_scores = bearing1_2_labeled["ch_h_rms"].to_numpy()

In [7]:
bearing1_2_metrics = compute_metrics(bearing1_2_y, bearing1_2_scores, frozen_threshold)
cross_rows = [{
    "dataset": "FEMTO Bearing1_2", "model": best_model, "threshold_type": "frozen_femto",
    "threshold": frozen_threshold, **bearing1_2_metrics,
}]

In [8]:
ims_feature_path = DATA_PROCESSED / "ims_1st_test_b1_features.parquet"
ims_run_path = IMS_DIR / "4. Bearings" / "1st_test"

if ims_feature_path.exists():
    ims_features = pd.read_parquet(ims_feature_path)
else:
    ims_run = load_ims_run(ims_run_path)
    ims_b1 = ims_run.rename(columns={"b1_ch1": "acc_h", "b1_ch2": "acc_v"})
    ims_features = extract_bearing_features(ims_b1[["acc_h", "acc_v", "file_idx"]], IMS_FS, WINDOW_SIZE, OVERLAP)
    ims_features.to_parquet(ims_feature_path, index=False)

In [9]:
ims_file_count = int(ims_features["file_idx"].max() + 1)
ims_healthy_end = int(ims_file_count * HEALTHY_RATIO)
ims_anomaly_start = int(ims_file_count * (HEALTHY_RATIO + TRANSITION_RATIO))
ims_mask = (ims_features["file_idx"] < ims_healthy_end) | (ims_features["file_idx"] >= ims_anomaly_start)
ims_labeled = ims_features.loc[ims_mask].copy()
ims_y = (ims_labeled["file_idx"] >= ims_anomaly_start).astype(int).to_numpy()
ims_scores = ims_labeled["ch_h_rms"].to_numpy()

In [10]:
ims_metrics = compute_metrics(ims_y, ims_scores, frozen_threshold)
cross_rows.append({
    "dataset": "IMS 1st_test bearing 1", "model": best_model, "threshold_type": "frozen_femto",
    "threshold": frozen_threshold, **ims_metrics,
})

In [11]:
bearing1_2_train_scores = bearing1_2.loc[bearing1_2["file_idx"] < healthy_end, "ch_h_rms"].to_numpy()
ims_train_scores = ims_features.loc[ims_features["file_idx"] < ims_healthy_end, "ch_h_rms"].to_numpy()

bearing1_2_threshold = float(np.mean(bearing1_2_train_scores) + ANOMALY_SIGMA * np.std(bearing1_2_train_scores))
ims_threshold = float(np.mean(ims_train_scores) + ANOMALY_SIGMA * np.std(ims_train_scores))

cross_rows.extend([
    {"dataset": "FEMTO Bearing1_2", "model": best_model, "threshold_type": "asset_calibrated", "threshold": bearing1_2_threshold, **compute_metrics(bearing1_2_y, bearing1_2_scores, bearing1_2_threshold)},
    {"dataset": "IMS 1st_test bearing 1", "model": best_model, "threshold_type": "asset_calibrated", "threshold": ims_threshold, **compute_metrics(ims_y, ims_scores, ims_threshold)},
])

In [12]:
cross_dataset_metrics = pd.DataFrame(cross_rows)
cross_dataset_metrics.to_csv(METRICS_DIR / "cross_dataset_metrics.csv", index=False)
cross_dataset_metrics.query("threshold_type == 'asset_calibrated'").to_csv(
    METRICS_DIR / "calibrated_threshold_metrics.csv", index=False
)
print(cross_dataset_metrics.to_string(index=False))

               dataset       model   threshold_type  threshold  roc_auc   pr_auc       f1  precision   recall  time_to_detection
      FEMTO Bearing1_2 Rolling RMS     frozen_femto   0.747281 0.782897 0.615057 0.394044   0.994220 0.245714           0.057000
IMS 1st_test bearing 1 Rolling RMS     frozen_femto   0.747281 0.688289 0.409480 0.000000   0.000000 0.000000                NaN
      FEMTO Bearing1_2 Rolling RMS asset_calibrated   0.524031 0.782897 0.615057 0.387234   0.758333 0.260000           0.209667
IMS 1st_test bearing 1 Rolling RMS asset_calibrated   0.181732 0.688289 0.409480 0.004382   0.973684 0.002196           4.767333


In [13]:
development_display = development_metrics.copy()
development_display.insert(0, "dataset", "FEMTO Bearing1_1")
development_display.insert(2, "threshold_type", "development")
final_comparison = pd.concat([development_display, cross_dataset_metrics], ignore_index=True, sort=False)
final_comparison.to_csv(METRICS_DIR / "final_comparison_metrics.csv", index=False)
plot_comparison_table(
    final_comparison[["dataset", "model", "threshold_type", "roc_auc", "pr_auc", "f1", "precision", "recall"]],
    TABLES_DIR / "final_model_comparison_table.png",
)

In [14]:
roc_results = {}
for dataset_name, labels, scores in [
    ("FEMTO Bearing1_2", bearing1_2_y, bearing1_2_scores),
    ("IMS 1st_test", ims_y, ims_scores),
]:
    fpr_array, tpr_array, _ = roc_curve(labels, scores)
    roc_results[dataset_name] = (fpr_array, tpr_array, roc_auc_score(labels, scores))

plot_roc_curves(roc_results, FIGURES_DIR / "frozen_rolling_rms_cross_dataset_roc_curves.png")

The best development model is Rolling RMS, so the frozen FEMTO threshold is applied directly to RMS-like scores first. The asset-calibrated comparison then uses only early healthy data to estimate a local mean + 3? threshold. FEMTO Bearing1_2 is the cleanest generalization test because it shares the same sensor and sampling context. IMS is evaluated with the same windowed feature pipeline on bearing 1 channel 1.